# Tier 4.2 — Agentic ReAct with Hybrid First Stage (Azure GPU)

**BSARD RAG Thesis | RQ1 | Backbone: react_hybrid_rrf_k60**

## This run: ReAct on hybrid_rrf_k60 first stage (test split)

One ReAct experiment on the **test split** (222 questions), using the T4.0-hybrid first stage
(`hybrid_rrf_k60`: BM25 k1=1.5 b=0.25 lemmatize text_only + mE5-large concat_2x, RRF k=60,
first_stage_k=100). Same ReAct loop parameters as `react_bm25_test`
(max_steps=5, top_k_shown=10, overlap_threshold=1.1, max_article_tokens=200, zero-shot D1).
First stage is the only axis of variation.

| Experiment | Backbone | Purpose |
|---|---|---|
| `react_hybrid_rrf_k60_test` | `hybrid_rrf_k60` | ReAct on T4.0-hybrid first stage — pool quality ablation |

## Ablation design

| Comparison | Measures |
|---|---|
| T4.2-hybrid vs `hybrid_rrf_k60` (T3-A) | Value of ReAct reasoning loop on hybrid candidate pool |
| T4.2-hybrid vs `react_bm25_test` (T4.2-BM25) | Pool quality: same ReAct loop, hybrid vs BM25 first stage |
| T4.2-hybrid vs `llm_rerank_binary_top50_hybrid_rrf_k60_test` (T4.0-hybrid) | Agentic vs non-agentic LLM on identical hybrid pool |

## Before running — one-time setup

1. **GPU compute instance**: Azure ML → Compute → Create → `Standard_NC4as_T4_v3`
2. **Set Cell 0** with your `GITHUB_TOKEN` and `AZURE_CONTAINER_SAS_URL`
3. **Prerequisites in blob storage** (upload locally before running):
   - `embeddings/intfloat_multilingual_e5_large_concat_2x.npy` (~87 MB)
   - `embeddings/intfloat_multilingual_e5_large_concat_2x_ids.npy` (~0.2 MB)
   - `results/hybrid/hybrid_rrf_k60_test.json` (significance anchor — T3-A)
   - `results/agentic/ReAct/react_bm25_test.json` (significance anchor — T4.2-BM25)
   - `results/agentic/llm_judge/llm_rerank/llm_rerank_binary_top50_hybrid_rrf_k60_test.json` (T4.0-hybrid anchor)
4. Run cells top to bottom

## Expected execution times (T4 GPU)

| Phase | Expected time |
|---|---|
| Setup (Cells 0–8) | ~25 min (embedding load adds ~1 min vs BM25-only) |
| TEST experiment (Cell 10) | ~70–80 min (similar to react_bm25 run; hybrid search ~285 ms/query overhead) |
| Significance tests (Cell 13) | ~5 min |
| **Total** | **~100 min** |

In [ ]:
# ── Cell 0: Configuration ─────────────────────────────────────────────────────
# Set GITHUB_TOKEN and AZURE_CONTAINER_SAS_URL before running any other cell.

GITHUB_TOKEN = ''
# How to get:
#   github.com → Settings → Developer settings
#   → Personal access tokens → Tokens (classic) → New token → scope: repo → Generate

AZURE_CONTAINER_SAS_URL = ''
# How to get:
#   Azure Portal → Storage Accounts → your account
#   → Containers → bsard-data → (...) → Generate SAS
#   → Permissions: Read + List + Write → Expiry: 1 year → Generate
#   → Copy the full "Blob SAS URL" (starts with https://...)

REPO     = 'MariusPasch/bsard-rag-thesis'
CLONE_DIR = '/home/azureuser/repo'                # mono-repo clone root
REPO_DIR  = f'{CLONE_DIR}/RQ1_Retrieval_Methods'  # RQ1 component root

assert GITHUB_TOKEN,            'Set GITHUB_TOKEN above before running!'
assert AZURE_CONTAINER_SAS_URL, 'Set AZURE_CONTAINER_SAS_URL above before running!'
print('Config OK')

In [ ]:
# ── Cell 1: Verify GPU ────────────────────────────────────────────────────────
import subprocess
result = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total,memory.free', '--format=csv,noheader'],
    capture_output=True, text=True
)
if result.stdout.strip():
    print('GPU:', result.stdout.strip())
    print('GPU OK')
else:
    print('WARNING: No GPU detected!')
    print('  Azure ML: ensure compute instance uses NC4as_T4_v3 or NC6ads_A10_v4')
    print(result.stderr)

In [ ]:
# ── Cell 2: Install Ollama and pull llama3.1:8b (~10 min on first run) ────────
import json, os, subprocess, time, urllib.request

OLLAMA_LOG = '/tmp/ollama_server.log'

def model_available() -> bool:
    try:
        with urllib.request.urlopen('http://localhost:11434/api/tags', timeout=5) as r:
            return any('llama3.1' in m['name'] for m in json.loads(r.read()).get('models', []))
    except Exception:
        return False

if not os.path.exists('/usr/local/bin/ollama'):
    print('Installing Ollama via official script...')
    subprocess.run(
        ['bash', '-c', 'curl -fsSL https://ollama.com/install.sh | sh'],
        check=True
    )
    print('Ollama installed.')
else:
    print('Ollama already installed.')

if not model_available():
    print('Starting Ollama server...')
    subprocess.Popen(
        ['ollama', 'serve'],
        env={
            **os.environ,
            'HOME': '/root',
            'OLLAMA_NUM_GPU': '99',
            'OLLAMA_FLASH_ATTENTION': '1',
            'OLLAMA_HOST': '0.0.0.0:11434',
        },
        stdout=open(OLLAMA_LOG, 'w'),
        stderr=subprocess.STDOUT,
    )
    time.sleep(8)
    if not model_available():
        print('Pulling llama3.1:8b (~4.7 GB, ~5-10 min)...')
        subprocess.run(['ollama', 'pull', 'llama3.1:8b'], check=True)
        time.sleep(3)

resp   = urllib.request.urlopen('http://localhost:11434/api/tags', timeout=10)
models = [m['name'] for m in json.loads(resp.read()).get('models', [])]
print('Available models:', models)
assert any('llama3.1' in m for m in models), 'llama3.1:8b not found!'
print('Ollama ready.')

In [ ]:
# ── Cell 3: Download data from Azure Blob Storage ─────────────────────────────
# Resumes partial downloads automatically (seeks to existing byte offset).
# Missing optional blobs show a warning, not an error.
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'azure-storage-blob', 'tqdm'],
               check=True)

from azure.storage.blob import ContainerClient
from pathlib import Path
from tqdm.auto import tqdm

OUTPUT_DIR  = Path(REPO_DIR) / 'output'
EMB_DIR     = OUTPUT_DIR / 'embeddings'
RESULTS_DIR = OUTPUT_DIR / 'results'
CACHE_DIR   = OUTPUT_DIR / 'cache'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
EMB_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)
(RESULTS_DIR / 'hybrid').mkdir(parents=True, exist_ok=True)
(RESULTS_DIR / 'agentic' / 'ReAct').mkdir(parents=True, exist_ok=True)
(RESULTS_DIR / 'agentic' / 'llm_judge' / 'llm_rerank').mkdir(parents=True, exist_ok=True)

EMB_SLUG = 'intfloat_multilingual_e5_large_concat_2x'
client   = ContainerClient.from_container_url(AZURE_CONTAINER_SAS_URL)


def _download_blob(blob_name: str, dest_path: Path, required: bool = True) -> bool:
    """Download blob → dest_path with tqdm progress and byte-level resume."""
    try:
        bc         = client.get_blob_client(blob_name)
        total_size = bc.get_blob_properties()['size']
    except Exception:
        tag = '[REQUIRED]' if required else '[optional]'
        print(f'  {tag} {blob_name} — not found in blob storage')
        if required:
            raise FileNotFoundError(f'Required blob missing: {blob_name}')
        return False

    existing = dest_path.stat().st_size if dest_path.exists() else 0
    if existing == total_size:
        print(f'  Already complete: {dest_path.name} ({total_size / 1e6:.1f} MB)')
        return True

    offset = existing if 0 < existing < total_size else 0
    mode   = 'ab' if offset > 0 else 'wb'
    if offset > 0:
        print(f'  Resuming {dest_path.name}: {offset/1e6:.1f}/{total_size/1e6:.1f} MB done')

    with tqdm(total=total_size, initial=offset, unit='B', unit_scale=True,
              desc=f'  {dest_path.name}', ncols=90, leave=True) as pbar:
        with open(dest_path, mode) as f:
            for chunk in bc.download_blob(offset=offset).chunks():
                f.write(chunk)
                pbar.update(len(chunk))
    return True


# ── Required: corpus + mE5-large embeddings ───────────────────────────────────
print('=== Required files ===')
_download_blob('bsard_articles_dedup.parquet', OUTPUT_DIR / 'bsard_articles_dedup.parquet')
_download_blob('bsard_corpus.db',              OUTPUT_DIR / 'bsard_corpus.db')
_download_blob(f'embeddings/{EMB_SLUG}.npy',      EMB_DIR / f'{EMB_SLUG}.npy')
_download_blob(f'embeddings/{EMB_SLUG}_ids.npy',  EMB_DIR / f'{EMB_SLUG}_ids.npy')

# ── Optional: BM25 tokenization disk cache (saves ~3 min on resume) ───────────
# Uploaded by Cell 16 at the end of any run (partial or complete).
print('\n=== Optional resume caches ===')
_download_blob(
    'cache/tokenized_lemmatize_text_only.pkl',
    CACHE_DIR / 'tokenized_lemmatize_text_only.pkl',
    required=False,
)
_download_blob(
    'results/agentic/ReAct/react_hybrid_rrf_k60_test_checkpoint.json',
    RESULTS_DIR / 'agentic' / 'ReAct' / 'react_hybrid_rrf_k60_test_checkpoint.json',
    required=False,
)

# ── Optional: significance anchors ────────────────────────────────────────────
print('\n=== Optional significance anchors ===')
_download_blob(
    'results/hybrid/hybrid_rrf_k60_test.json',
    RESULTS_DIR / 'hybrid' / 'hybrid_rrf_k60_test.json',
    required=False,
)
_download_blob(
    'results/agentic/ReAct/react_bm25_test.json',
    RESULTS_DIR / 'agentic' / 'ReAct' / 'react_bm25_test.json',
    required=False,
)
_download_blob(
    'results/agentic/llm_judge/llm_rerank/llm_rerank_binary_top50_hybrid_rrf_k60_test.json',
    RESULTS_DIR / 'agentic' / 'llm_judge' / 'llm_rerank' /
    'llm_rerank_binary_top50_hybrid_rrf_k60_test.json',
    required=False,
)

print('\nAll downloads complete.')
# Show resume state
ckpt = RESULTS_DIR / 'agentic' / 'ReAct' / 'react_hybrid_rrf_k60_test_checkpoint.json'
bm25_cache = CACHE_DIR / 'tokenized_lemmatize_text_only.pkl'
if ckpt.exists():
    import json
    data = json.loads(ckpt.read_text())
    n_done = len(data.get('completed', []))
    acc_wall = data.get('accumulated_wall_s', 0)
    print(f'\nCheckpoint found: {n_done}/222 questions done ({acc_wall/60:.1f} min accumulated).')
    print('Cell 10 will resume from where this left off.')
else:
    print('\nNo checkpoint found — Cell 10 will start fresh.')
if bm25_cache.exists():
    print(f'BM25 cache found ({bm25_cache.stat().st_size / 1e6:.1f} MB) — tokenization will load from disk.')
else:
    print('No BM25 cache — first run will tokenize corpus (~3 min) and cache it.')

In [ ]:
# ── Cell 4: Clone GitHub repo ─────────────────────────────────────────────────
import os, subprocess

if os.path.exists(CLONE_DIR):
    print('Repo already cloned — pulling latest...')
    subprocess.run(
        ['git', '-C', CLONE_DIR, 'remote', 'set-url', 'origin',
         f'https://{GITHUB_TOKEN}@github.com/{REPO}.git'],
        capture_output=True, text=True
    )
    r = subprocess.run(['git', '-C', CLONE_DIR, 'pull'], capture_output=True, text=True)
    print(r.stdout.strip() or r.stderr.strip())
else:
    print(f'Cloning {REPO}...')
    r = subprocess.run(
        ['git', 'clone', f'https://{GITHUB_TOKEN}@github.com/{REPO}.git', CLONE_DIR],
        capture_output=True, text=True, timeout=120
    )
    if r.returncode != 0:
        print('STDERR:', r.stderr[-1000:])
        raise RuntimeError('git clone failed')

os.chdir(REPO_DIR)
branch = subprocess.run(['git', 'branch', '--show-current'],
                        capture_output=True, text=True).stdout.strip()
print(f'Working directory: {os.getcwd()}  |  branch: {branch}')

In [ ]:
# ── Cell 5: Install Python dependencies (~5 min) ──────────────────────────────
import subprocess, sys, os

cuda_out = subprocess.run(['nvcc', '--version'], capture_output=True, text=True).stdout
cuda_ver = 'cu118' if 'release 11' in cuda_out else 'cu121'
print(f'CUDA detected → torch variant: {cuda_ver}')

cmds = [
    ([sys.executable, '-m', 'pip', 'install', '-q', '-r', f'{REPO_DIR}/requirements.txt'],
     'requirements.txt'),
    ([sys.executable, '-m', 'pip', 'install', '-q', 'azure-storage-blob'],
     'azure-storage-blob'),
    ([sys.executable, '-m', 'pip', 'install', '-q', 'torch',
      '--index-url', f'https://download.pytorch.org/whl/{cuda_ver}'],
     'torch'),
    ([sys.executable, '-m', 'pip', 'install', '-q',
      'langgraph>=0.1.0', 'langchain-core>=0.2.0', 'requests'],
     'langgraph + langchain-core + requests'),
    ([sys.executable, '-m', 'pip', 'install', '-q', 'tf-keras'],
     'tf-keras'),
    ([sys.executable, '-m', 'pip', 'install', '-q', 'timm>=0.9.2'],
     'timm>=0.9.2'),
]
for cmd, label in cmds:
    print(f'  {label} ...', end='', flush=True)
    r = subprocess.run(cmd, capture_output=True, text=True)
    print(' OK' if r.returncode == 0 else f' WARN({r.returncode})')
    if r.returncode != 0:
        print(r.stderr[-200:])

# bsard_evaluation — editable local package from the RQ3 repo
# bsard_evaluation lives in the same mono-repo (RQ3_Autonomous_Evaluation),
# already cloned above — just install it editable.
RQ3_DIR = f'{CLONE_DIR}/RQ3_Autonomous_Evaluation'
print('  bsard_evaluation ...', end='', flush=True)
r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', RQ3_DIR],
                   capture_output=True, text=True)
print(' OK' if r.returncode == 0 else f' WARN({r.returncode})\n{r.stderr[-200:]}')

In [ ]:
# ── Cell 6: Install spaCy French model ───────────────────────────────────────
# BM25 lemmatization (first stage of hybrid) requires fr_core_news_lg.
import subprocess, sys
import spacy as _spacy

_sv   = _spacy.__version__
_base = 'https://github.com/explosion/spacy-models/releases/download'
_whl  = f'fr_core_news_lg-{_sv}/fr_core_news_lg-{_sv}-py3-none-any.whl'
r = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', f'{_base}/{_whl}'],
    capture_output=True, text=True
)
if r.returncode != 0:
    _sv2  = '.'.join(_sv.split('.')[:2]) + '.0'
    _whl2 = f'fr_core_news_lg-{_sv2}/fr_core_news_lg-{_sv2}-py3-none-any.whl'
    r = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q', f'{_base}/{_whl2}'],
        capture_output=True, text=True
    )
    if r.returncode != 0:
        raise RuntimeError(f'spaCy fr_core_news_lg install failed:\n{r.stderr[-400:]}')

import spacy
spacy.load('fr_core_news_lg')
print(f'spaCy {spacy.__version__} OK — fr_core_news_lg loaded')

In [ ]:
# ── Cell 7: Pre-flight checks + LLM latency benchmark ────────────────────────
import os, sys, time
import requests as _req
from pathlib import Path

os.chdir(REPO_DIR)

EMB_SLUG = 'intfloat_multilingual_e5_large_concat_2x'

# ── File checks ───────────────────────────────────────────────────────────────
for p in [
    Path('output/bsard_articles_dedup.parquet'),
    Path('output/bsard_corpus.db'),
    Path('evaluation/data/fewshot_examples.json'),
    Path(f'output/embeddings/{EMB_SLUG}.npy'),
    Path(f'output/embeddings/{EMB_SLUG}_ids.npy'),
]:
    print(f'  {"OK     " if p.exists() else "MISSING"}  {p}')

# ── Ollama alive ──────────────────────────────────────────────────────────────
try:
    r      = _req.get('http://localhost:11434/api/tags', timeout=5)
    models = [m['name'] for m in r.json().get('models', [])]
    print(f'\nOllama alive. Models: {models}')
except Exception as e:
    raise RuntimeError(f'Ollama not reachable: {e}')

# ── LLM latency benchmark (binary prompt — same as D1 scoring in ReAct) ──────
TEST_PROMPT = (
    'Question : Quelles sont les conditions pour obtenir un congé parental ?\n\n'
    'Passage : Le congé parental est accordé aux travailleurs salariés ayant un '
    'enfant de moins de 12 ans.\n\n'
    'Le passage est-il pertinent pour répondre à la question ?\n'
    'Répondez uniquement par « Oui » ou « Non ».\n\nPertinent :'
)

def llm_generate(prompt, max_tokens=5):
    t0   = time.perf_counter()
    resp = _req.post('http://localhost:11434/api/generate', json={
        'model': 'llama3.1:8b', 'prompt': prompt, 'stream': False,
        'options': {'temperature': 0.0, 'num_predict': max_tokens},
    }, timeout=300)
    return resp.json()['response'].strip(), (time.perf_counter() - t0) * 1000

print('\nBenchmarking (3 binary calls — simulates D1 scoring)...')
lats = []
for i in range(3):
    resp, lat = llm_generate(TEST_PROMPT)
    lats.append(lat)
    print(f'  Call {i+1}: {resp!r:10s}  {lat:.0f} ms')

warm_lat = sum(lats[1:]) / 2
mean_lat = sum(lats) / len(lats)
print(f'\nWarm mean (calls 2–3): {warm_lat:.0f} ms  |  Overall mean: {mean_lat:.0f} ms')
if warm_lat < 5000:
    print('GPU confirmed (fast).')
elif warm_lat < 30000:
    print('MARGINAL — verify GPU in Cell 1.')
else:
    print('WARNING: likely on CPU. Re-check nvidia-smi.')

# ── Runtime estimate ──────────────────────────────────────────────────────────
# T4.2-BM25 actual: 18.5s/query (D1=7s + generate=11.4s + retrieval=0.1s)
# T4.2-hybrid: same D1+generate; hybrid retrieval adds ~57ms/step × 5 = ~285ms/query
D1_POOL_EST = 12     # empirical pool size from BM25 run (~12 articles)
GEN_STEPS   = 5      # max_steps
est_d1_s    = D1_POOL_EST * max(warm_lat, 400) / 1000
est_gen_s   = GEN_STEPS * 2.3  # ~2.3s per generate step (warm)
est_retr_s  = GEN_STEPS * 0.057  # ~57ms per hybrid search step
est_total_s = est_d1_s + est_gen_s + est_retr_s
print(f'\n--- Estimated experiment time (T4.2-hybrid, max_steps=5, top_k_shown=10) ---')
print(f'  D1 reranking: ~{D1_POOL_EST} articles × {max(warm_lat,400):.0f}ms = {est_d1_s:.0f}s')
print(f'  Generate:     {GEN_STEPS} steps × ~2.3s = {est_gen_s:.1f}s')
print(f'  Retrieval:    {GEN_STEPS} steps × ~57ms hybrid = {est_retr_s:.1f}s')
print(f'  Total/query:  ~{est_total_s:.0f}s  →  222 questions: ~{222*est_total_s/60:.0f} min')

In [ ]:
# ── Cell 8: Hardcoded hyperparameters — Round-2 (§16.5) ─────────────────────
# Round-2 changes vs Round-1 (preserved as the _bkp notebook):
#   max_steps           5  →  8
#   top_k_shown         10 →  3
#   overlap_threshold   1.1→  0.6  (now query-token Jaccard, not ID-Jaccard)
#   max_article_tokens  200→  300
#   max_tokens_generate 64 →  128
#   + use_function_calling=True, use_action_regex_v2=True,
#     use_invalid_action_echo=True, use_fewshot_trajectories=True,
#     use_gap_prompt=True, inject_step_budget=True,
#     overlap_metric="query_tokens",
#     seed_search_k=20, topup_threshold=20, topup_k=50, search_snippet_chars=80
# Estimated runtime: ~30-40 s/query × 222 ≈ ~2-2.5 h on Tesla T4.

MAX_STEPS                = 8
TOP_K_SHOWN              = 3
OVERLAP_THRESHOLD        = 0.6
MAX_ARTICLE_TOKENS       = 300
MAX_TOKENS_GENERATE      = 128
SEED_SEARCH_K            = 20
TOPUP_THRESHOLD          = 20
TOPUP_K                  = 50
SEARCH_SNIPPET_CHARS     = 80
USE_FUNCTION_CALLING     = True
USE_ACTION_REGEX_V2      = True
USE_INVALID_ACTION_ECHO  = True
USE_FEWSHOT_TRAJECTORIES = True
USE_GAP_PROMPT           = True
INJECT_STEP_BUDGET       = True
OVERLAP_METRIC           = "query_tokens"

print('Round-2 hyperparameters:')
for k in ('MAX_STEPS', 'TOP_K_SHOWN', 'OVERLAP_THRESHOLD', 'MAX_ARTICLE_TOKENS',
         'MAX_TOKENS_GENERATE', 'SEED_SEARCH_K', 'TOPUP_THRESHOLD', 'TOPUP_K',
         'SEARCH_SNIPPET_CHARS', 'USE_FUNCTION_CALLING', 'USE_ACTION_REGEX_V2',
         'USE_INVALID_ACTION_ECHO', 'USE_FEWSHOT_TRAJECTORIES',
         'USE_GAP_PROMPT', 'INJECT_STEP_BUDGET', 'OVERLAP_METRIC'):
    print(f'  {k:25s} = {globals()[k]}')
print(f'\nEstimated test runtime: ~{222 * 35 / 60:.0f} min ({222} questions × ~35 s/query)')


In [ ]:
# ── Cell 9: Verify hybrid retriever can be built ──────────────────────────────
# Sanity check: load embeddings and run one test query before the full experiment.
# This also warms up the mE5-large model in VRAM so the first experiment query
# does not pay a cold-load penalty inside the timed loop.
import os, sys, subprocess, time
from pathlib import Path
import pandas as pd

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)

from retrieval.sparse import BM25Retriever
from retrieval.dense import DenseRetriever
from retrieval.hybrid import HybridRetriever

def _vram():
    r = subprocess.run(
        ['nvidia-smi', '--query-gpu=memory.used,memory.free', '--format=csv,noheader'],
        capture_output=True, text=True
    )
    return r.stdout.strip()

print('Loading corpus...')
corpus = pd.read_parquet('output/bsard_articles_dedup.parquet')
print(f'  {len(corpus):,} articles')

# ── Unload Ollama to free system RAM for BM25 tokenization ───────────────────
print('\nUnloading Ollama model to free RAM for BM25 tokenization...')
subprocess.run(['ollama', 'stop', 'llama3.1:8b'], capture_output=True)
print('Ollama model unloaded.')

# ── BM25 ──────────────────────────────────────────────────────────────────────
print('\nBuilding BM25Retriever (lemmatize, text_only, k1=1.5, b=0.25)...')
t0     = time.perf_counter()
sparse = BM25Retriever(
    corpus, variant='okapi', normalization='lemmatize',
    field_weighting='text_only', k1=1.5, b=0.25,
)
print(f'  BM25 ready in {time.perf_counter() - t0:.1f}s')

# ── DenseRetriever ────────────────────────────────────────────────────────────
EMB_SLUG = 'intfloat_multilingual_e5_large_concat_2x'
print(f'\nBuilding DenseRetriever (mE5-large, concat_2x)...')
print(f'  VRAM before: {_vram()}')
t0    = time.perf_counter()
dense = DenseRetriever(
    corpus,
    model_name='intfloat/multilingual-e5-large',
    field_weighting='concat_2x',
    passage_prefix='passage: ',
    query_prefix='query: ',
    device='cuda',
    embeddings_dir=Path('output/embeddings'),
)
build_src = dense._index_build_source  # 'disk' | 'encode'
print(f'  DenseRetriever ready in {time.perf_counter() - t0:.1f}s  (source: {build_src})')
print(f'  VRAM after:  {_vram()}')

# ── HybridRetriever ───────────────────────────────────────────────────────────
print('\nBuilding HybridRetriever (RRF k=60, first_stage_k=100)...')
hybrid_retriever = HybridRetriever(
    sparse_retriever=sparse,
    dense_retriever=dense,
    fusion_method='rrf',
    rrf_k=60,
    first_stage_k=100,
)

# Sanity check query
print('\nSanity check (warm-up query)...')
t0 = time.perf_counter()
_ids, _lat = hybrid_retriever.retrieve('Quelles sont les conditions du contrat de travail ?', top_k=10)
warmup_s = time.perf_counter() - t0
print(f'  {len(_ids)} IDs in {_lat:.0f}ms (wall {warmup_s:.1f}s)')
print(f'  First 5: {_ids[:5]}')

_ids2, _lat2 = hybrid_retriever.retrieve('Quelle est la peine pour vol simple ?', top_k=10)
print(f'  Steady-state latency: {_lat2:.0f}ms')

print('\nHybrid retriever verified. Ready for Cell 10.')
print('NOTE: The experiment script (Cell 10) rebuilds the hybrid retriever internally.')
print('      This cell is a pre-flight sanity check only — del vars to free memory.')

del sparse, dense, hybrid_retriever, corpus
import gc; gc.collect()
print('Memory freed.')

In [ ]:
# ── Cell 10: TEST experiment — react_hybrid_rrf_k60_test_v2 (Round-2) ────────
# Writes to output/results/agentic/ReAct/round2/ with _v2 suffix on filenames
# so the Round-1 results in the parent dir are never overwritten.
#
# Writes (on completion):
#   output/results/agentic/ReAct/round2/react_hybrid_rrf_k60_test_v2.json
#   output/results/agentic/ReAct/round2/react_hybrid_rrf_k60_test_v2_traces.json
# Checkpoint (deleted on success):
#   output/results/agentic/ReAct/round2/react_hybrid_rrf_k60_test_v2_checkpoint.json

import json, os, subprocess, sys, time
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)

from bsard_evaluation import EvaluationHarness, filter_by as _bsard_filter_by
from bsard_evaluation.config import TierConfig
from evaluation.runner import (
    _to_trec_run, _to_trec_qrels, _trec_metrics_to_legacy, save_result,
)
from evaluation.split import load_questions
from evaluation.stratify import load_strata
from retrieval.agentic.llm_client import OllamaClient
from retrieval.agentic.llm_eval_prompts import load_fewshot_examples
from retrieval.agentic.react import ReActRetriever
from retrieval.dense import DenseRetriever
from retrieval.hybrid import HybridRetriever
from retrieval.sparse import BM25Retriever

_K_FULL     = [1, 5, 10, 20, 50, 100, 200, 500]
ROUND2_DIR  = Path(f'{REPO_DIR}/output/results/agentic/ReAct/round2')
RESULT_PATH = ROUND2_DIR / 'react_hybrid_rrf_k60_test_v2.json'
TRACES_PATH = ROUND2_DIR / 'react_hybrid_rrf_k60_test_v2_traces.json'
CKPT_PATH   = ROUND2_DIR / 'react_hybrid_rrf_k60_test_v2_checkpoint.json'
ROUND2_DIR.mkdir(parents=True, exist_ok=True)

# ── Early exit if already complete ────────────────────────────────────────────
if RESULT_PATH.exists():
    r10 = json.loads(RESULT_PATH.read_text())['metrics'].get('Recall@10', 0)
    print(f'Already done. R@10={r10:.4f}  (delete {RESULT_PATH.name} to rerun)')
else:
    # ── Load checkpoint ────────────────────────────────────────────────────────
    checkpoint: dict = {}
    accumulated_wall_s: float = 0.0
    if CKPT_PATH.exists():
        ckpt_data = json.loads(CKPT_PATH.read_text())
        checkpoint = {e['question_id']: e for e in ckpt_data.get('completed', [])}
        accumulated_wall_s = float(ckpt_data.get('accumulated_wall_s', 0.0))
        print(f'Resuming from checkpoint: {len(checkpoint)}/222 questions done '
              f'({accumulated_wall_s/60:.1f} min accumulated).')
    else:
        print('No checkpoint — starting fresh.')

    # ── Load corpus ────────────────────────────────────────────────────────────
    print('\nLoading corpus...')
    df = pd.read_parquet(f'{REPO_DIR}/output/bsard_articles_dedup.parquet')
    print(f'  {len(df):,} articles')

    # ── Build hybrid retriever ─────────────────────────────────────────────────
    print('\nUnloading Ollama to free RAM for BM25 tokenization...')
    subprocess.run(['ollama', 'stop', 'llama3.1:8b'], capture_output=True)

    print('Building BM25 (lemmatize, text_only, k1=1.5, b=0.25)...')
    t0     = time.time()
    sparse = BM25Retriever(df, variant='okapi', normalization='lemmatize',
                           field_weighting='text_only', k1=1.5, b=0.25)
    print(f'  BM25 ready in {time.time()-t0:.1f}s')

    print('Building DenseRetriever (mE5-large concat_2x, loading from disk)...')
    t0    = time.time()
    dense = DenseRetriever(
        df,
        model_name='intfloat/multilingual-e5-large',
        field_weighting='concat_2x',
        passage_prefix='passage: ',
        query_prefix='query: ',
        device='cuda',
        embeddings_dir=Path(f'{REPO_DIR}/output/embeddings'),
    )
    print(f'  DenseRetriever ready in {time.time()-t0:.1f}s  '
          f'(source: {getattr(dense, "_index_build_source", "?")})')

    hybrid_backbone = HybridRetriever(
        sparse_retriever=sparse, dense_retriever=dense,
        fusion_method='rrf', rrf_k=60, first_stage_k=100,
    )
    print('HybridRetriever ready.')

    # ── Build ReAct retriever (Round-2) ────────────────────────────────────────
    fewshot = load_fewshot_examples()
    llm     = OllamaClient()
    lat_pre = llm.preflight()
    print(f'\nOllama preflight OK ({lat_pre:.0f} ms)')

    react = ReActRetriever(
        retriever=hybrid_backbone,
        df_articles=df,
        llm_client=llm,
        top_k_shown=TOP_K_SHOWN,
        search_snippet_chars=SEARCH_SNIPPET_CHARS,
        max_steps=MAX_STEPS,
        overlap_threshold=OVERLAP_THRESHOLD,
        fewshot_examples=fewshot,
        max_article_tokens=MAX_ARTICLE_TOKENS,
        # Round-2 Block A
        max_tokens_generate=MAX_TOKENS_GENERATE,
        use_function_calling=USE_FUNCTION_CALLING,
        use_action_regex_v2=USE_ACTION_REGEX_V2,
        use_invalid_action_echo=USE_INVALID_ACTION_ECHO,
        use_fewshot_trajectories=USE_FEWSHOT_TRAJECTORIES,
        # Round-2 Block B
        overlap_metric=OVERLAP_METRIC,
        use_gap_prompt=USE_GAP_PROMPT,
        inject_step_budget=INJECT_STEP_BUDGET,
        # Round-2 Block D
        seed_search_k=SEED_SEARCH_K,
        topup_threshold=TOPUP_THRESHOLD,
        topup_k=TOPUP_K,
    )

    print('Warming up LLM (loading into VRAM)...', end=' ', flush=True)
    llm.generate('Prêt.', max_tokens=1)
    print('done')

    # ── Load questions ─────────────────────────────────────────────────────────
    questions = load_questions(subset='test')
    strata    = load_strata()
    remaining = [q for q in questions if q['question_id'] not in checkpoint]
    print(f'\nRunning {len(remaining)} remaining questions (of {len(questions)} total)...')
    print(f'Estimated time: ~{len(remaining) * 35 / 60:.0f} min at ~35 s/query\n')

    # ── Main question loop (checkpointed) ──────────────────────────────────────
    t_session = time.time()

    with tqdm(total=len(questions), initial=len(checkpoint),
              desc='questions', unit='q', ncols=90) as pbar:
        for q in remaining:
            ranked, latency_ms = react.retrieve(q['question_text'], top_k=100)
            trace      = react.get_all_traces()[-1]
            lat_bd     = react.get_latency_breakdown()
            topup_flag = react.get_pool_topup_flags()[-1]

            checkpoint[q['question_id']] = {
                'question_id':          q['question_id'],
                'ranked':               ranked,
                'latency_ms':           latency_ms,
                'trace':                trace,
                'latency_breakdown_ms': lat_bd,
                'pool_topped_up':       topup_flag,
            }

            session_wall = time.time() - t_session
            CKPT_PATH.write_text(json.dumps(
                {
                    'experiment_id':      'react_hybrid_rrf_k60_test_v2',
                    'accumulated_wall_s': round(accumulated_wall_s + session_wall, 1),
                    'completed':          list(checkpoint.values()),
                },
                ensure_ascii=False,
            ))

            n_steps = len(trace)
            pbar.set_postfix({'lat': f'{latency_ms:.0f}ms', 'steps': n_steps,
                              'topup': '*' if topup_flag else ''})
            pbar.update(1)

    session_wall = time.time() - t_session
    total_wall   = accumulated_wall_s + session_wall
    print(f'\nAll {len(questions)} questions done. Total wall clock: {total_wall/60:.1f} min')

    # ── Reconstruct ordered arrays ─────────────────────────────────────────────
    results_dict = {q['question_id']: checkpoint[q['question_id']]['ranked']            for q in questions}
    latencies    = [checkpoint[q['question_id']]['latency_ms']                          for q in questions]
    all_traces_  = [checkpoint[q['question_id']]['trace']                               for q in questions]
    all_bds_     = [checkpoint[q['question_id']]['latency_breakdown_ms']                for q in questions]
    all_topups_  = [checkpoint[q['question_id']].get('pool_topped_up', False)           for q in questions]
    ground_truth = {q['question_id']: q['relevant_article_ids']                        for q in questions}

    # ── Compute metrics ────────────────────────────────────────────────────────
    trec_run   = _to_trec_run(results_dict)
    trec_qrels = _to_trec_qrels(ground_truth)
    harness    = EvaluationHarness(TierConfig(tiers=[0, 1, 2], custom_k=_K_FULL))
    lat_dict   = {str(q['question_id']): checkpoint[q['question_id']]['latency_ms'] for q in questions}

    trec_metrics = harness.evaluate(
        qrels=trec_qrels, run=trec_run, latencies=lat_dict,
        queries={str(q['question_id']): q['question_text'] for q in questions},
        verbose=False,
    )
    metrics = _trec_metrics_to_legacy(trec_metrics)
    if 'MAP@100' in metrics and 'MAP' not in metrics:
        metrics['MAP'] = metrics['MAP@100']

    arr = np.array(latencies)
    latency_distribution = {
        'mean': float(arr.mean()), 'std':  float(arr.std()),
        'p50':  float(np.percentile(arr, 50)), 'p90': float(np.percentile(arr, 90)),
        'p95':  float(np.percentile(arr, 95)), 'p99': float(np.percentile(arr, 99)),
        'min':  float(arr.min()),              'max': float(arr.max()),
        'index_build_s': 0.0,
    }

    str_strata = {str(k): v for k, v in strata.items()}
    def _eval_stratum(field, value):
        sub_qrels = _bsard_filter_by(trec_qrels, str_strata, field, value)
        if not sub_qrels: return {}
        sub_run = {qid: trec_run[qid] for qid in sub_qrels if qid in trec_run}
        return _trec_metrics_to_legacy(harness.evaluate(qrels=sub_qrels, run=sub_run, verbose=False))

    stratified = {
        'single_article':           _eval_stratum('article_count', 'single_article'),
        'multi_article':            _eval_stratum('article_count', 'multi_article'),
        'lexically_aligned':        _eval_stratum('lex_align', 'lexically_aligned'),
        'semantically_paraphrased': _eval_stratum('lex_align', 'semantically_paraphrased'),
        'with_cross_refs':          _eval_stratum('cross_ref', 'with_cross_refs'),
        'without_cross_refs':       _eval_stratum('cross_ref', 'without_cross_refs'),
    }

    # ── Loop stats from checkpoint traces ──────────────────────────────────────
    n = len(questions)
    steps_per_query     = [len(t) for t in all_traces_]
    finished            = sum(1 for t in all_traces_ if any(e.get('tool_name') == 'finish' for e in t))
    terminated_by_limit = sum(
        1 for t in all_traces_
        if t and not any(e.get('tool_name') == 'finish' for e in t) and len(t) >= MAX_STEPS
    )
    total_overlap   = sum(sum(1 for e in t if e.get('overlap_guard_fired', False)) for t in all_traces_)
    total_preflight = sum(sum(1 for e in t if e.get('preflight_failed',   False)) for t in all_traces_)
    n_topup         = sum(1 for f in all_topups_ if f)

    loop_stats = {
        'n_queries':                            n,
        'mean_steps_per_query':                 float(np.mean(steps_per_query)),
        'fraction_queries_converged_finish':    finished / n,
        'fraction_queries_terminated_by_limit': terminated_by_limit / n,
        'total_overlap_guard_fires':            total_overlap,
        'total_preflight_failures':             total_preflight,
        'fraction_pool_topped_up':              n_topup / n,    # Round-2 Block D
    }

    keys = ('llm_generate', 'retrieval', 'rerank_posthoc')
    latency_breakdown_mean = {
        k: round(float(np.mean([b.get(k, 0.0) for b in all_bds_])), 1) for k in keys
    }

    # ── Assemble + save result ─────────────────────────────────────────────────
    result = {
        'experiment_id':   'react_hybrid_rrf_k60_test_v2',
        'timestamp':       time.strftime('%Y-%m-%dT%H:%M:%S'),
        'model_or_method': 'ReActRetriever',
        'hyperparameters': {
            'retrieval_backbone':       'hybrid_rrf_k60',
            'llm_backbone':             'llama3.1:8b',
            'max_steps':                MAX_STEPS,
            'top_k_shown':              TOP_K_SHOWN,
            'overlap_threshold':        OVERLAP_THRESHOLD,
            'overlap_metric':           OVERLAP_METRIC,
            'max_article_tokens':       MAX_ARTICLE_TOKENS,
            'max_tokens_generate':      MAX_TOKENS_GENERATE,
            'search_snippet_chars':     SEARCH_SNIPPET_CHARS,
            'seed_search_k':            SEED_SEARCH_K,
            'topup_threshold':          TOPUP_THRESHOLD,
            'topup_k':                  TOPUP_K,
            'use_function_calling':     USE_FUNCTION_CALLING,
            'use_action_regex_v2':      USE_ACTION_REGEX_V2,
            'use_invalid_action_echo':  USE_INVALID_ACTION_ECHO,
            'use_fewshot_trajectories': USE_FEWSHOT_TRAJECTORIES,
            'use_gap_prompt':           USE_GAP_PROMPT,
            'inject_step_budget':       INJECT_STEP_BUDGET,
            'd1_reranking':             'llama3.1:8b_binary',
            'round':                    2,
        },
        'preprocessing':   {'normalization': 'lemmatize', 'field_weighting': 'concat_2x',
                            'embedding_prefix': 'query/passage'},
        'token_length_audit':             {'fraction_truncated': 0.0, 'max_tokens_observed': 0},
        'training_regime':                'zero_shot',
        'latency_ms_mean':                latency_distribution['mean'],
        'latency_ms_std':                 latency_distribution['std'],
        'latency_distribution':           latency_distribution,
        'metrics':                        metrics,
        'significance_vs_anchor':         {'p_value_recall10': None, 'significant': None},
        'stratified':                     stratified,
        'agent_loop_stats':               loop_stats,
        'latency_breakdown_ms_mean':      latency_breakdown_mean,
        'total_experiment_wall_clock_s':  round(total_wall, 1),
        '_trec_run':   trec_run,
        '_trec_qrels': trec_qrels,
    }

    saved = save_result(result, results_dir=ROUND2_DIR)

    # ── Save traces ────────────────────────────────────────────────────────────
    traces_out = [
        {'question_id': q['question_id'],
         'trace':       checkpoint[q['question_id']]['trace'],
         'pool_topped_up': checkpoint[q['question_id']].get('pool_topped_up', False)}
        for q in questions
    ]
    TRACES_PATH.write_text(json.dumps(traces_out, ensure_ascii=False))

    # ── Tear down checkpoint on success ────────────────────────────────────────
    CKPT_PATH.unlink(missing_ok=True)
    print(f'\nSaved → {RESULT_PATH.name}')
    print(f'Saved → {TRACES_PATH.name}')
    print(f'\nR@10 = {metrics.get("Recall@10", 0):.4f}   '
          f'pool_topped_up on {n_topup}/{n} queries '
          f'({n_topup/n:.1%})')


In [ ]:
# ── Cell 11: Final results summary (Round-2) ─────────────────────────────────
import json
from pathlib import Path

ROUND2_DIR = Path(f'{REPO_DIR}/output/results/agentic/ReAct/round2')

print(f'{"Experiment":<45} {"R@10":>7} {"R@100":>7} {"MRR@10":>8} {"Wall(min)":>10}')
print('=' * 81)
for path in [
    Path(f'{REPO_DIR}/output/results/agentic/ReAct/react_bm25_test.json'),
    Path(f'{REPO_DIR}/output/results/agentic/ReAct/react_hybrid_rrf_k60_test.json'),
    ROUND2_DIR / 'react_hybrid_rrf_k60_test_v2.json',
]:
    if not path.exists():
        print(f'  {path.stem:<43}  (not yet run)')
        continue
    r    = json.loads(path.read_text())
    m    = r.get('metrics', {})
    wall = r.get('total_experiment_wall_clock_s', 0) / 60
    print(f'  {r["experiment_id"]:<43}'
          f'{m.get("Recall@10",0):>7.4f}'
          f'{m.get("Recall@100",0):>7.4f}'
          f'{m.get("MRR@10",0):>8.4f}'
          f'{wall:>10.1f}')

print('\n--- Round-2 hyperparameters ---')
print(f'  max_steps={MAX_STEPS}  top_k_shown={TOP_K_SHOWN}  '
      f'overlap_threshold={OVERLAP_THRESHOLD} ({OVERLAP_METRIC})  '
      f'max_article_tokens={MAX_ARTICLE_TOKENS}  '
      f'seed_search_k={SEED_SEARCH_K}  topup_threshold={TOPUP_THRESHOLD}')

print('\n--- Round-2 loop stats ---')
v2_path = ROUND2_DIR / 'react_hybrid_rrf_k60_test_v2.json'
if v2_path.exists():
    r    = json.loads(v2_path.read_text())
    loop = r.get('agent_loop_stats', {})
    bd   = r.get('latency_breakdown_ms_mean', {})
    lim  = loop.get('fraction_queries_terminated_by_limit', 0)
    topup = loop.get('fraction_pool_topped_up', 0)
    print(f'  mean_steps={loop.get("mean_steps_per_query",0):.1f}  '
          f'fin%={loop.get("fraction_queries_converged_finish",0):.1%}  '
          f'term_lim%={lim:.1%}  '
          f'pool_topup%={topup:.1%}')
    print(f'  preflight_fails={loop.get("total_preflight_failures",0)}  '
          f'overlap_fires={loop.get("total_overlap_guard_fires",0)}')
    print(f'  llm_gen={bd.get("llm_generate",0):.0f}ms  '
          f'retrieval={bd.get("retrieval",0):.0f}ms  '
          f'rerank={bd.get("rerank_posthoc",0):.0f}ms')


In [ ]:
# ── Cell 12: Per-step recall analysis (Round-2) ──────────────────────────────
# For each step t=1..MAX_STEPS, reconstructs the cumulative observed-ID pool and
# computes Recall@10, Recall@all_observed, MRR@10.
# Reads : react_hybrid_rrf_k60_test_v2_traces.json (round2/)
# Writes: react_hybrid_rrf_k60_test_v2_step_recall.json (round2/)
import json, sys
from pathlib import Path

sys.path.insert(0, REPO_DIR)
from evaluation.split import load_questions
from evaluation.metrics import evaluate

ROUND2_DIR  = Path(f'{REPO_DIR}/output/results/agentic/ReAct/round2')
TRACES_PATH = ROUND2_DIR / 'react_hybrid_rrf_k60_test_v2_traces.json'
OUTPUT_PATH = ROUND2_DIR / 'react_hybrid_rrf_k60_test_v2_step_recall.json'
RESULT_PATH = ROUND2_DIR / 'react_hybrid_rrf_k60_test_v2.json'

if not TRACES_PATH.exists():
    print('ERROR: traces file not found. Re-run Cell 10 first.')
else:
    traces_data  = json.loads(TRACES_PATH.read_text())
    questions    = load_questions(subset='test')
    ground_truth = {q['question_id']: q['relevant_article_ids'] for q in questions}
    n_queries    = len(traces_data)

    per_step = {}
    for step_t in range(1, MAX_STEPS + 1):
        step_results = {}
        pool_sizes   = []
        for entry in traces_data:
            qid  = entry['question_id']
            pool, seen = [], set()
            for t_entry in entry['trace'][:step_t]:
                for aid in t_entry.get('ids_returned', []):
                    if aid not in seen:
                        pool.append(aid)
                        seen.add(aid)
            step_results[qid] = pool
            pool_sizes.append(len(pool))

        m = evaluate(step_results, ground_truth)

        total_recall_all = 0.0
        for entry in traces_data:
            qid    = entry['question_id']
            pool_s = set(step_results[qid])
            rel    = set(ground_truth.get(qid, []))
            if rel:
                total_recall_all += len(pool_s & rel) / len(rel)
        mean_recall_all = total_recall_all / n_queries

        per_step[step_t] = {
            'Recall@10':           round(m.get('Recall@10', 0.0), 6),
            'Recall@all_observed': round(mean_recall_all,          6),
            'MRR@10':              round(m.get('MRR@10',    0.0), 6),
            'mean_pool_size':      round(sum(pool_sizes) / n_queries, 2),
        }

    final_d1 = {}
    if RESULT_PATH.exists():
        fm = json.loads(RESULT_PATH.read_text()).get('metrics', {})
        final_d1 = {
            'Recall@10': round(fm.get('Recall@10', 0.0), 6),
            'MRR@10':    round(fm.get('MRR@10',    0.0), 6),
        }

    print(f'{"Step":>6}  {"R@10":>8}  {"R@all":>8}  {"MRR@10":>8}  {"Pool":>6}')
    print('-' * 50)
    for step_t in range(1, MAX_STEPS + 1):
        sm = per_step[step_t]
        print(f'{step_t:>6}  {sm["Recall@10"]:>8.4f}  '
              f'{sm["Recall@all_observed"]:>8.4f}  '
              f'{sm["MRR@10"]:>8.4f}  '
              f'{sm["mean_pool_size"]:>6.1f}')
    if final_d1:
        print('-' * 50)
        print(f'{"D1":>6}  {final_d1["Recall@10"]:>8.4f}  '
              f'{"":>8}  {final_d1["MRR@10"]:>8.4f}')

    output = {
        'experiment_id': 'react_hybrid_rrf_k60_test_v2',
        'n_queries':      n_queries,
        'max_steps':      MAX_STEPS,
        'per_step':       per_step,
        'final_d1':       final_d1,
    }
    OUTPUT_PATH.write_text(json.dumps(output, indent=2, ensure_ascii=False))
    print(f'\nSaved: {OUTPUT_PATH}')


In [ ]:
# ── Cell 13: Significance tests (three-way) — Round-2 ────────────────────────
# Primary:     T4.2-hybrid v2 vs hybrid_rrf_k60 (T3-A) — ReAct loop value
# Secondary A: T4.2-hybrid v2 vs react_bm25_test (T4.2-BM25) — pool quality
# Secondary B: T4.2-hybrid v2 vs T4.0-hybrid — agentic vs non-agentic LLM
import json, os, sys
import numpy as np
from pathlib import Path
from scipy.stats import ttest_rel
from bsard_evaluation import per_query_recall as _bsard_per_query_recall

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)

REACT_DIR  = Path(f'{REPO_DIR}/output/results/agentic/ReAct')
ROUND2_DIR = REACT_DIR / 'round2'
HYBRID_DIR = Path(f'{REPO_DIR}/output/results/hybrid')
LLM_DIR    = Path(f'{REPO_DIR}/output/results/agentic/llm_judge/llm_rerank')

REACT_V2_PATH  = ROUND2_DIR / 'react_hybrid_rrf_k60_test_v2.json'
T3A_PATH       = HYBRID_DIR / 'hybrid_rrf_k60_test.json'
BM25_REACT_PATH = REACT_DIR / 'react_bm25_test.json'
T40_HYB_PATH   = LLM_DIR    / 'llm_rerank_binary_top50_hybrid_rrf_k60_test.json'

if not REACT_V2_PATH.exists():
    print('T4.2-hybrid v2 result not found — run Cell 10 first.')
else:
    react_v2 = json.loads(REACT_V2_PATH.read_text(encoding='utf-8'))
    if '_trec_run' not in react_v2 or '_trec_qrels' not in react_v2:
        raise RuntimeError('_trec_run/_trec_qrels missing — re-run Cell 10')

    r_v2_k10  = _bsard_per_query_recall(react_v2['_trec_qrels'], react_v2['_trec_run'], 10)
    r_v2_k100 = _bsard_per_query_recall(react_v2['_trec_qrels'], react_v2['_trec_run'], 100)
    print(f'T4.2-hybrid v2  R@10 = {np.mean(r_v2_k10):.4f}  '
          f'R@100 = {np.mean(r_v2_k100):.4f}')

    react_v2.setdefault('secondary_significance', {})

    print('\n' + '='*60)
    print('PRIMARY: T4.2-hybrid v2 vs hybrid_rrf_k60 (T3-A)')
    if T3A_PATH.exists():
        t3a = json.loads(T3A_PATH.read_text(encoding='utf-8'))
        r_t3a_k10  = _bsard_per_query_recall(t3a['_trec_qrels'], t3a['_trec_run'], 10)
        r_t3a_k100 = _bsard_per_query_recall(t3a['_trec_qrels'], t3a['_trec_run'], 100)
        if r_t3a_k10 is not None:
            _, p10  = ttest_rel(r_v2_k10,  r_t3a_k10)
            _, p100 = ttest_rel(r_v2_k100, r_t3a_k100)
            sig_str = 'SIGNIFICANT (p<0.05)' if p10 < 0.05 else 'not significant'
            print(f'  T4.2-hybrid v2      R@10 = {np.mean(r_v2_k10):.4f}')
            print(f'  hybrid_rrf_k60 T3-A R@10 = {np.mean(r_t3a_k10):.4f}')
            print(f'  Delta R@10  = {np.mean(r_v2_k10) - np.mean(r_t3a_k10):+.4f}')
            print(f'  Delta R@100 = {np.mean(r_v2_k100) - np.mean(r_t3a_k100):+.4f}')
            print(f'  p-value R@10  = {p10:.4f}  → {sig_str}')
            print(f'  p-value R@100 = {p100:.4f}')
            react_v2['significance_vs_anchor'] = {
                'anchor_experiment_id': 'hybrid_rrf_k60_test',
                'p_value_recall10':  round(float(p10),  4),
                'p_value_recall100': round(float(p100), 4),
                'significant':       bool(p10 < 0.05),
            }
    else:
        print(f'  [WARN] {T3A_PATH.name} not found')

    print('\nSECONDARY A: T4.2-hybrid v2 vs react_bm25_test (T4.2-BM25)')
    if BM25_REACT_PATH.exists():
        bm25_react = json.loads(BM25_REACT_PATH.read_text(encoding='utf-8'))
        r_bm25_k10  = _bsard_per_query_recall(bm25_react['_trec_qrels'], bm25_react['_trec_run'], 10)
        r_bm25_k100 = _bsard_per_query_recall(bm25_react['_trec_qrels'], bm25_react['_trec_run'], 100)
        if r_bm25_k10 is not None:
            _, p10s  = ttest_rel(r_v2_k10,  r_bm25_k10)
            _, p100s = ttest_rel(r_v2_k100, r_bm25_k100)
            sig_str = 'SIGNIFICANT (p<0.05)' if p10s < 0.05 else 'not significant'
            print(f'  T4.2-hybrid v2 R@10 = {np.mean(r_v2_k10):.4f}')
            print(f'  T4.2-BM25      R@10 = {np.mean(r_bm25_k10):.4f}')
            print(f'  Delta R@10  = {np.mean(r_v2_k10) - np.mean(r_bm25_k10):+.4f}')
            print(f'  p-value R@10  = {p10s:.4f}  → {sig_str}')
            react_v2['secondary_significance']['vs_react_bm25_test'] = {
                'anchor_experiment_id': 'react_bm25_test',
                'p_value_recall10':  round(float(p10s),  4),
                'p_value_recall100': round(float(p100s), 4),
                'significant':       bool(p10s < 0.05),
            }
    else:
        print(f'  [WARN] {BM25_REACT_PATH.name} not found')

    print('\nSECONDARY B: T4.2-hybrid v2 vs T4.0-hybrid')
    if T40_HYB_PATH.exists():
        t40_hyb = json.loads(T40_HYB_PATH.read_text(encoding='utf-8'))
        r_t40_k10  = _bsard_per_query_recall(t40_hyb['_trec_qrels'], t40_hyb['_trec_run'], 10)
        r_t40_k100 = _bsard_per_query_recall(t40_hyb['_trec_qrels'], t40_hyb['_trec_run'], 100)
        if r_t40_k10 is not None:
            _, p10t  = ttest_rel(r_v2_k10,  r_t40_k10)
            _, p100t = ttest_rel(r_v2_k100, r_t40_k100)
            sig_str = 'SIGNIFICANT (p<0.05)' if p10t < 0.05 else 'not significant'
            print(f'  T4.2-hybrid v2  R@10 = {np.mean(r_v2_k10):.4f}')
            print(f'  T4.0-hybrid     R@10 = {np.mean(r_t40_k10):.4f}')
            print(f'  Delta R@10  = {np.mean(r_v2_k10) - np.mean(r_t40_k10):+.4f}')
            print(f'  p-value R@10  = {p10t:.4f}  → {sig_str}')
            react_v2['secondary_significance']['vs_llm_rerank_binary_top50_hybrid_rrf_k60_test'] = {
                'anchor_experiment_id': 'llm_rerank_binary_top50_hybrid_rrf_k60_test',
                'p_value_recall10':  round(float(p10t),  4),
                'p_value_recall100': round(float(p100t), 4),
                'significant':       bool(p10t < 0.05),
            }
    else:
        print(f'  [WARN] {T40_HYB_PATH.name} not found')

    REACT_V2_PATH.write_text(
        json.dumps(react_v2, ensure_ascii=False, indent=2), encoding='utf-8'
    )
    print(f'\nPatched {REACT_V2_PATH.name} with significance results.')


In [ ]:
# ── Cell 14: Full comparison table (T3-A through T4.2-hybrid v2) ─────────────
import json
from pathlib import Path

REACT_DIR  = Path(f'{REPO_DIR}/output/results/agentic/ReAct')
ROUND2_DIR = REACT_DIR / 'round2'
HYBRID_DIR = Path(f'{REPO_DIR}/output/results/hybrid')
LLM_DIR    = Path(f'{REPO_DIR}/output/results/agentic/llm_judge/llm_rerank')

rows = [
    ('hybrid_rrf_k60 (T3-A, no rerank)',
     HYBRID_DIR / 'hybrid_rrf_k60_test.json'),
    ('T4.0-hybrid (LLM rerank, binary)',
     LLM_DIR / 'llm_rerank_binary_top50_hybrid_rrf_k60_test.json'),
    ('T4.2-BM25 (ReAct bm25, Round-1)',
     REACT_DIR / 'react_bm25_test.json'),
    ('T4.2-hybrid v1 (ReAct hybrid_rrf_k60, Round-1)',
     REACT_DIR / 'react_hybrid_rrf_k60_test.json'),
    ('T4.2-hybrid v2 (ReAct hybrid_rrf_k60, Round-2)',
     ROUND2_DIR / 'react_hybrid_rrf_k60_test_v2.json'),
]

print(f'{"System":<55}  R@10    R@100   MRR@10  Lat(ms)')
print('=' * 100)
for label, path in rows:
    if not path.exists():
        print(f'  {label:<53}  (not found)')
        continue
    d   = json.loads(path.read_text())
    m   = d.get('metrics', {})
    lat = d.get('latency_ms_mean', 0)
    print(f'  {label:<53}  {m.get("Recall@10",0):.4f}  '
          f'{m.get("Recall@100",0):.4f}  {m.get("MRR@10",0):.4f}  {lat:.0f}')

# ReAct-specific loop stats — Round-1 v1 + Round-2 v2
for label, path in [
    ('react_bm25_test',                  REACT_DIR / 'react_bm25_test.json'),
    ('react_hybrid_rrf_k60_test (v1)',   REACT_DIR / 'react_hybrid_rrf_k60_test.json'),
    ('react_hybrid_rrf_k60_test_v2',     ROUND2_DIR / 'react_hybrid_rrf_k60_test_v2.json'),
]:
    if not path.exists(): continue
    d    = json.loads(path.read_text())
    loop = d.get('agent_loop_stats', {})
    sig  = d.get('significance_vs_anchor', {})
    if loop:
        print(f'\nLoop stats ({label}):')
        print(f'  mean_steps                  : {loop.get("mean_steps_per_query",0):.1f}')
        print(f'  finish convergence          : {loop.get("fraction_queries_converged_finish",0):.1%}')
        print(f'  terminated by step limit    : {loop.get("fraction_queries_terminated_by_limit",0):.1%}')
        if "fraction_pool_topped_up" in loop:
            print(f'  pool topped up              : {loop.get("fraction_pool_topped_up",0):.1%}')
        if sig:
            anchor = sig.get('anchor_experiment_id', '?')
            p10    = sig.get('p_value_recall10', None)
            sigstr = 'significant' if sig.get('significant') else 'not significant'
            p_str  = f'p={p10:.4f}' if p10 is not None else 'not computed'
            print(f'  Significance vs {anchor}: {sigstr} ({p_str})')


In [ ]:
# ── Cell 15: Commit and push Round-2 results to GitHub ───────────────────────
# Stages the Round-2 v2 JSONs under output/results/agentic/ReAct/round2/.
# Round-1 result files in the parent dir are already committed.
import os, subprocess
from pathlib import Path

GIT_NAME  = 'MariusPasch'
GIT_EMAIL = 'paschalidismarios@gmail.com'

def git(args):
    return subprocess.run(['git'] + args, cwd=REPO_DIR, capture_output=True, text=True)

git(['config', 'user.email', GIT_EMAIL])
git(['config', 'user.name',  GIT_NAME])
git(['remote', 'set-url', 'origin', f'https://{GITHUB_TOKEN}@github.com/{REPO}.git'])

files_to_stage = [
    'output/results/agentic/ReAct/round2/react_hybrid_rrf_k60_test_v2.json',
    'output/results/agentic/ReAct/round2/react_hybrid_rrf_k60_test_v2_traces.json',
    'output/results/agentic/ReAct/round2/react_hybrid_rrf_k60_test_v2_step_recall.json',
]
staged = [f for f in files_to_stage if Path(f'{REPO_DIR}/{f}').exists()]
for f in staged:
    git(['add', '-f', f])  # force-add since output/ is gitignored

print(f'Staged {len(staged)} file(s):')
for f in staged:
    print(f'  {f}')

if not staged:
    print('No result files found to commit.')
else:
    status = git(['status', '--short']).stdout.strip()
    if not status:
        print('Nothing new to commit.')
    else:
        commit = git(['commit', '-m',
            'T4.2-hybrid Round-2 ReAct results — full §16 stack on hybrid_rrf_k60\n\n'
            ''])
        print('Commit:', commit.stdout.strip() or commit.stderr.strip())
        push = git(['push', 'origin', 'main'])
        if push.returncode != 0:
            print('Push failed — trying pull --rebase first...')
            git(['pull', '--rebase', 'origin', 'main'])
            push2 = git(['push', 'origin', 'main'])
            print('Pushed.' if push2.returncode == 0 else f'Push failed: {push2.stderr[-400:]}')
        else:
            print('Pushed.')


In [ ]:
# ── Cell 17: v1 vs v2 side-by-side (Round-1 vs Round-2) ──────────────────────
# Prints once both result files exist. Pulls overall + stratified R@10 and the
# loop diagnostics most affected by Round-2 (preflight failures, mean steps,
# pool top-up rate).
import json
from pathlib import Path

V1 = Path(f'{REPO_DIR}/output/results/agentic/ReAct/react_hybrid_rrf_k60_test.json')
V2 = Path(f'{REPO_DIR}/output/results/agentic/ReAct/round2/react_hybrid_rrf_k60_test_v2.json')

if not V1.exists() or not V2.exists():
    missing = [p.name for p in (V1, V2) if not p.exists()]
    print(f'Side-by-side: missing {missing} — re-run prior cells first.')
else:
    r1 = json.loads(V1.read_text())
    r2 = json.loads(V2.read_text())

    print(f'{"Metric":<42} {"v1":>10} {"v2":>10} {"Δ":>10}')
    print('=' * 75)
    for k in ('Recall@10', 'Recall@100', 'MRR@10', 'NDCG@10'):
        a = r1['metrics'].get(k, 0.0)
        b = r2['metrics'].get(k, 0.0)
        print(f'  {k:<40}{a:>10.4f}{b:>10.4f}{b-a:>10.4f}')

    print()
    print('Stratified R@10:')
    for stratum in ('single_article', 'multi_article',
                    'lexically_aligned', 'semantically_paraphrased'):
        a = r1.get('stratified', {}).get(stratum, {}).get('Recall@10', 0.0)
        b = r2.get('stratified', {}).get(stratum, {}).get('Recall@10', 0.0)
        print(f'  {stratum:<40}{a:>10.4f}{b:>10.4f}{b-a:>10.4f}')

    print()
    print('Loop diagnostics:')
    l1 = r1.get('agent_loop_stats', {})
    l2 = r2.get('agent_loop_stats', {})
    for k, fmt in [
        ('mean_steps_per_query',                 '{:.2f}'),
        ('fraction_queries_converged_finish',    '{:.1%}'),
        ('fraction_queries_terminated_by_limit', '{:.1%}'),
        ('total_preflight_failures',             '{:d}'),
        ('total_overlap_guard_fires',            '{:d}'),
        ('fraction_pool_topped_up',              '{:.1%}'),
    ]:
        a = l1.get(k, 0)
        b = l2.get(k, 0)
        print(f'  {k:<40}{fmt.format(a):>10}{fmt.format(b):>10}')

    print()
    print('Latency (ms/query mean):')
    for k in ('llm_generate', 'retrieval', 'rerank_posthoc'):
        a = r1.get('latency_breakdown_ms_mean', {}).get(k, 0)
        b = r2.get('latency_breakdown_ms_mean', {}).get(k, 0)
        print(f'  {k:<40}{a:>10.0f}{b:>10.0f}{b-a:>10.0f}')


In [ ]:
# ── Cell 16: Upload results + caches to Azure Blob Storage ────────────────────
# Uploads results, traces, step-recall, BM25 disk cache, and checkpoint (if present).
# Run this cell at any point — after partial progress or after full completion.
# The BM25 cache and checkpoint are downloaded in Cell 3 on the next VM session,
# so a full VM restart loses nothing.
from pathlib import Path
from azure.storage.blob import ContainerClient

REACT_DIR  = Path(f'{REPO_DIR}/output/results/agentic/ReAct')
CACHE_DIR  = Path(f'{REPO_DIR}/output/cache')
client     = ContainerClient.from_container_url(AZURE_CONTAINER_SAS_URL)
uploaded   = []


def _upload(local_path: Path, blob_name: str, label: str = '') -> None:
    if not local_path.exists():
        print(f'  (skip) {local_path.name} — not found')
        return
    sz = local_path.stat().st_size
    print(f'  Uploading {label or local_path.name} ({sz/1e6:.1f} MB) ...', end='', flush=True)
    with open(local_path, 'rb') as f:
        client.get_blob_client(blob_name).upload_blob(f, overwrite=True)
    print(' done')
    uploaded.append(blob_name)


# ── Experiment result files ────────────────────────────────────────────────────
print('=== Result files ===')
for json_file in sorted(REACT_DIR.glob('react_hybrid_rrf_k60_*.json')):
    _upload(json_file, f'results/agentic/ReAct/{json_file.name}')

# ── Resume caches — upload even on partial completion ─────────────────────────
print('\n=== Resume caches (for cross-VM-restart recovery) ===')
_upload(
    CACHE_DIR / 'tokenized_lemmatize_text_only.pkl',
    'cache/tokenized_lemmatize_text_only.pkl',
    'BM25 tokenization cache',
)
_upload(
    REACT_DIR / 'react_hybrid_rrf_k60_test_checkpoint.json',
    'results/agentic/ReAct/react_hybrid_rrf_k60_test_checkpoint.json',
    'question checkpoint',
)

print(f'\nUploaded {len(uploaded)} file(s) to blob storage.')
if not any('react_hybrid_rrf_k60_test.json' in u and 'checkpoint' not in u for u in uploaded):
    ckpt = REACT_DIR / 'react_hybrid_rrf_k60_test_checkpoint.json'
    if ckpt.exists():
        import json as _json
        data   = _json.loads(ckpt.read_text())
        n_done = len(data.get('completed', []))
        acc    = data.get('accumulated_wall_s', 0)
        print(f'\nPartial progress saved: {n_done}/222 questions ({acc/60:.1f} min).')
        print('On the next VM session, Cell 3 will restore the checkpoint and Cell 10 will resume.')
    else:
        print('\nNOTE: final result JSON not present. Run Cell 10 to completion first.')
print('\nDownload locally via Azure Storage Explorer → output/results/agentic/ReAct/')